# Import libraries

In [3]:
import shutil
from functools import partial
from multiprocessing import Pool
from pathlib import Path
from pprint import pprint

import cv2
import fitz
import numpy as np
import pandas as pd
import pymupdf as fitz
import torch
from datasets import load_dataset
from matplotlib import pyplot as plt
from natsort import natsorted
from PIL import Image
from tqdm.auto import tqdm
from transformers.utils.import_utils import is_flash_attn_2_available

/home/thuy0050/mg61_scratch2/thuy0050/conda/envs/colpali/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import msgspec
from pathlib import Path
from msgspec.json import format
from pprint import pformat, pprint

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(format(encoder.encode(data)))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)


def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()

    output = []

    with open(file_path, "rb") as file:
        data = file.read()
        if jsonl:
            output = decoder.decode_lines(data)
        else:
            output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


# Get a ranking subset from `marqo-gs-dataset`

For each dataset such as marqo_gs_full_10m, there are 4 splits as discussed before.

- `query_0_product_id_0` represents in-domain set,
- `query_1_product_id_0` represents novel query set,
- `query_0_product_id_1` represents novel document set,
- `query_1_product_id_1` represents zero shot set

For each split, there is a ground truth csv containing triplet information, a set of validation ground truth and a set of test ground truth.
`score_reciprocal = 100 / (score + 1)`

In [5]:
dataset_root = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset")
image_folder = dataset_root / "images_wfash"
metadata_folder = dataset_root / "marqo_gs_wfash_1m"

mapping_dict = {
    "in_domain": "query_0_product_id_0.csv",
    "novel_query": "query_1_product_id_0.csv",
    "novel_doc": "query_0_product_id_1.csv",
    "zeroshot_doc": "query_1_product_id_1.csv",
    "corpus1": "corpus_1.json",
    "corpus2": "corpus_2.json"
}

In [95]:
in_domain = pd.read_csv(metadata_folder / mapping_dict["in_domain"], index_col=False)
novel_query = pd.read_csv(metadata_folder / mapping_dict["novel_query"], index_col=False)
novel_doc = pd.read_csv(metadata_folder / mapping_dict["novel_doc"], index_col=False)
zeroshot_doc = pd.read_csv(metadata_folder / mapping_dict["zeroshot_doc"], index_col=False)
corpus1 = read_json(metadata_folder / mapping_dict['corpus1'])
corpus2 = read_json(metadata_folder / mapping_dict['corpus2'])

# corpus1_fashion_5m = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/marqo_gs_fashion_5m/corpus_1.json")
# corpus2_fashion_5m = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/marqo_gs_fashion_5m/corpus_2.json")

The file is of type: <class 'dict'>
The file contains 516307 items.
The file is of type: <class 'dict'>
The file contains 516225 items.


In [96]:
# Rename the image paths

# def rename_image_local(x):
#     temp = image_folder / Path(x).name
#     if temp.is_file():
#         return str(temp)
#     return ""

# in_domain['image_local'] = in_domain['image_local'].apply(lambda x: rename_image_local(x))
# novel_query['image_local'] = novel_query['image_local'].apply(lambda x: rename_image_local(x))
# novel_doc['image_local'] = novel_doc['image_local'].apply(lambda x: rename_image_local(x))
# zeroshot_doc['image_local'] = zeroshot_doc['image_local'].apply(lambda x: rename_image_local(x))

# in_domain.to_csv(metadata_folder / mapping_dict["in_domain"])
# novel_query.to_csv(metadata_folder / mapping_dict["novel_query"])
# novel_doc.to_csv(metadata_folder / mapping_dict["novel_doc"])
# zeroshot_doc.to_csv(metadata_folder / mapping_dict["zeroshot_doc"])

In [ ]:
corpus1_wfash_1m_set_int = set([str(x) for x in list(corpus1.keys())])
# corpus1_fashion_5m_product_id_set_int = set([int(x) for x in list(corpus1_fashion_5m.keys())])
# corpus2_fashion_5m_product_id_set_int = set([int(x) for x in list(corpus2_fashion_5m.keys())])

# dev_queries_in_domain_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/marqo_gs_fashion_5m/query_0_product_id_0_gt_dev.json")
# test_queries_in_domain_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/marqo_gs_fashion_5m/query_0_product_id_0_gt_test.json")
# dev_queries_novel_query_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/marqo_gs_fashion_5m/query_0_product_id_1_gt_dev.json")
# test_queries_novel_query_json = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/marqo_gs_fashion_5m/query_0_product_id_1_gt_test.json")

# query_json_mapping_dict = {
#     "in_domain_dev": dev_queries_in_domain_json,
#     "in_domain_test": test_queries_in_domain_json,
#     "novel_query_dev": dev_queries_novel_query_json,
#     "novel_query_test": test_queries_novel_query_json
# }

# in_list = []
# not_in_list = []

# new_query_list_mapping_dict = {
#     "in_domain_dev": [],
#     "in_domain_test": [],
#     "novel_query_dev": [],
#     "novel_query_test": []
# }

# for k, v in query_json_mapping_dict.items():
#     for query, i in v.items():
#         query_list = []
#         for product_id, score in i.items():
#             if str(product_id) in corpus1_wfash_1m_set_int:
#                 query_list.append({product_id: score})
#         if len(query_list) > 0:
#             new_query_list_mapping_dict[k].append(query_list)
            
# for k, v in new_query_list_mapping_dict.items():
#     print(k, len(v))

## Woman Fashion 1M passage

In [6]:
dataset_root = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset")
image_folder = dataset_root / "images_wfash"
metadata_folder = dataset_root / "marqo_gs_wfash_1m"

mapping_dict = {
    "in_domain": "query_0_product_id_0.csv",
    "novel_query": "query_1_product_id_0.csv",
    "novel_doc": "query_0_product_id_1.csv",
    "zeroshot_doc": "query_1_product_id_1.csv",
    "corpus1": "corpus_1.json",
    "corpus2": "corpus_2.json"
}

in_domain = pd.read_csv(metadata_folder / mapping_dict["in_domain"])
in_domain = in_domain[["query_id", "query", "product_id", "image_local", "position"]]
in_domain = in_domain.rename(columns={
    "query_id": "query_id",
    "query": "query",
    "product_id": "positive_document_ids",
    "image_local": "source",
    "position": "rank"
})
in_domain["query_id"] = in_domain["query_id"].astype(str)
in_domain["positive_document_ids"] = in_domain["positive_document_ids"].astype(str)
in_domain["source"] = in_domain["source"].apply(lambda x: "/".join(x.split("/")[-2:]))

print(len(set(in_domain["positive_document_ids"].astype(str).unique())), len(corpus1_wfash_1m_set_int), len(set(in_domain["positive_document_ids"].astype(str).unique()).intersection(corpus1_wfash_1m_set_int)))
in_domain = in_domain.groupby("query_id").agg({"query": "first", "positive_document_ids": list, "rank": list, "source": list}).reset_index()
from datasets import load_dataset, load_dataset_builder, Image, DatasetDict, Value, Sequence, Dataset

in_domain_dataset = Dataset.from_pandas(in_domain)

NameError: name 'corpus1_wfash_1m_set_int' is not defined

In [127]:
novel_query = pd.read_csv(metadata_folder / mapping_dict["novel_query"])
novel_query = novel_query[["query_id", "query", "product_id", "image_local", "position"]]
novel_query = novel_query.rename(columns={
    "query_id": "query_id",
    "query": "query",
    "product_id": "positive_document_ids",
    "image_local": "source",
    "position": "rank"
})
novel_query["query_id"] = novel_query["query_id"].astype(str)
novel_query["positive_document_ids"] = novel_query["positive_document_ids"].astype(str)
novel_query["source"] = novel_query["source"].apply(lambda x: "/".join(x.split("/")[-2:]))
print(len(set(novel_query["positive_document_ids"].astype(str).unique())), len(corpus1_wfash_1m_set_int), len(set(novel_query["positive_document_ids"].astype(str).unique()).intersection(corpus1_wfash_1m_set_int)))

novel_query = novel_query.groupby("query_id").agg({"query": "first", "positive_document_ids": list, "rank": list, "source": list}).reset_index()
from datasets import load_dataset, load_dataset_builder, Image, DatasetDict, Value, Sequence, Dataset

novel_query_dataset = Dataset.from_pandas(novel_query)

131699 516307 131699


In [131]:
hf_marqo_gs_wfash_1m_dataset = DatasetDict({
    "train": in_domain_dataset,
    "test": novel_query_dataset
})

In [132]:
hf_marqo_gs_wfash_1m_dataset.push_to_hub("LouisDo2108/marqo_gs_wfash_1m_tevatron", private=True)

Creating parquet from Arrow format: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.60ba/s]
Processing Files (1 / 1): 100%|█████████████████████████████████████████████████████████████| 43.9MB / 43.9MB, 7.31MB/s  
New Data Upload: 100%|██████████████████████████████████████████████████████████████████████| 43.9MB / 43.9MB, 7.31MB/s  
Creating parquet from Arrow format: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00, 33.06ba/s]
Processing Files (1 / 1): 100%|█████████████████████████████████████████████████████████████| 11.1MB / 11.1MB, 3.97MB/s  
New Data Upload: 100%|██████████████████████████████████████████████████████████████████████| 11.1MB / 11.1MB, 3.97MB/s  
Uploading the dataset shards: 100%|███████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.78s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/LouisDo2108/marqo_gs_wfash_1m_tevatron/commit/5597710775b70bb1fc7888f09a18b848138bc84f', commit_message='Upload dataset', commit_description='', oid='5597710775b70bb1fc7888f09a18b848138bc84f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/LouisDo2108/marqo_gs_wfash_1m_tevatron', endpoint='https://huggingface.co', repo_type='dataset', repo_id='LouisDo2108/marqo_gs_wfash_1m_tevatron'), pr_revision=None, pr_num=None)

## Woman Fashion 1M corpus

In [7]:
dataset_root = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset")
image_folder = dataset_root / "images_wfash"
metadata_folder = dataset_root / "marqo_gs_wfash_1m"

mapping_dict = {
    "in_domain": "query_0_product_id_0.csv",
    "novel_query": "query_1_product_id_0.csv",
    "novel_doc": "query_0_product_id_1.csv",
    "zeroshot_doc": "query_1_product_id_1.csv",
    "corpus1": "corpus_1.json",
    "corpus2": "corpus_2.json"
}

in_domain = pd.read_csv(metadata_folder / mapping_dict["in_domain"])
in_domain = in_domain[["query_id", "query", "product_id", "image_local", "position"]]
in_domain = in_domain.rename(columns={
    "query_id": "query_id",
    "query": "query",
    "product_id": "positive_document_ids",
    "image_local": "source",
    "position": "rank"
})
in_domain["query_id"] = in_domain["query_id"].astype(str)
in_domain["positive_document_ids"] = in_domain["positive_document_ids"].astype(str)
in_domain["source"] = in_domain["source"].apply(lambda x: "/".join(x.split("/")[-2:]))

# print(len(set(in_domain["positive_document_ids"].astype(str).unique())), len(corpus1_wfash_1m_set_int), len(set(in_domain["positive_document_ids"].astype(str).unique()).intersection(corpus1_wfash_1m_set_int)))
# in_domain = in_domain.groupby("query_id").agg({"query": "first", "positive_document_ids": list, "rank": list, "source": list}).reset_index()
# from datasets import load_dataset, load_dataset_builder, Image, DatasetDict, Value, Sequence, Dataset

# in_domain_dataset = Dataset.from_pandas(in_domain)

# in_domain = pd.read_csv(metadata_folder / mapping_dict["in_domain"])[["product_id", "image_local"]]
# novel_query = pd.read_csv(metadata_folder / mapping_dict["novel_query"])[["product_id", "image_local"]]
# corpus = pd.concat([in_domain, novel_query], axis=0).drop_duplicates().reset_index(drop=True)
# corpus.rename(columns={
#     "product_id": "docid",
#     "image_local": "source",
# }, inplace=True)
# corpus["source"] = corpus["source"].apply(lambda x: "/".join(x.split("/")[-2:]))
# corpus['docid'] = corpus['docid'].astype(str)
# from PIL import Image
# corpus["image"] = corpus["source"].apply(lambda x: Image.open(f"/home/thuy0050/mg61_scratch2/thuy0050/data/marqo-gs-dataset/{x}"))
# corpus = Dataset.from_pandas(corpus)
# # corpus.push_to_hub("LouisDo2108/marqo_gs_wfash_1m_corpus_tevatron")

In [8]:
in_domain

,query_id,query,positive_document_ids,source,rank
0,0,Earmuffs,11950591053179551937,images_wfash/11950591053179551937.webp,2
1,0,Earmuffs,13060356563414168615,images_wfash/13060356563414168615.webp,3
2,0,Earmuffs,6741082963333937131,images_wfash/6741082963333937131.webp,5
3,0,Earmuffs,8848678524883684053,images_wfash/8848678524883684053.webp,8
4,0,Earmuffs,11334298244441157208,images_wfash/11334298244441157208.webp,12
...,...,...,...,...,...
596581,98234,Eternity Wedding Bands,11745996422324990500,images_wfash/11745996422324990500.webp,94
596582,98234,Eternity Wedding Bands,707761173130550843,images_wfash/707761173130550843.webp,95
596583,98234,Eternity Wedding Bands,16745652588244524515,images_wfash/16745652588244524515.webp,96
596584,98234,Eternity Wedding Bands,9474091397466115898,images_wfash/9474091397466115898.webp,97


In [10]:
print(1)

1
